# Reproducing the public release

This notebook shows how the safe, aggregate-only release is read and how each public figure is produced. It uses only files under `public_data/`; no student-level records or survey responses are loaded.

The original student-level workflow is documented separately because its raw inputs are restricted.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image

ROOT = Path.cwd()
if not (ROOT / "public_data").exists():
    ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PUBLIC = ROOT / "public_data"
FIGURE_DATA = PUBLIC / "figure_source_data"
FIGURES = ROOT / "reproduced_figures"
FIGURES.mkdir(exist_ok=True)
print(f"Repository: {ROOT}")
print(f"Public data: {PUBLIC}")
print(f"Figure output: {FIGURES}")


## Function-to-file map

The first group of functions creates aggregate CSV files. The second group creates the PNG figures.


In [ ]:
file_map = pd.DataFrame([
    ["prepare_safe_public_release.build_activity_summary", "public_data/activity_summary.csv"],
    ["prepare_safe_public_release.build_questionnaire_summary", "public_data/questionnaire_summary_counts_percentages.csv"],
    ["prepare_safe_public_release.build_reported_difficulties_summary", "public_data/reported_difficulties_summary.csv"],
    ["prepare_safe_public_release.copy_csv", "public_data/question_level_facility_index.csv"],
    ["prepare_safe_public_release.copy_csv", "public_data/response_status_summary_by_quiz_question.csv"],
    ["prepare_safe_public_release.copy_csv", "public_data/invalid_input_frequency_table.csv"],
    ["plot_public_data.save_activity", "reproduced_figures/activity_mean_grade.png"],
    ["plot_public_data.save_activity_participation_performance", "reproduced_figures/activity_participation_performance.png"],
    ["plot_public_data.save_facility", "reproduced_figures/question_level_facility_index.png"],
    ["plot_public_data.save_status", "reproduced_figures/response_status_summary.png"],
    ["plot_public_data.save_response_status_by_question", "reproduced_figures/response_status_by_question.png"],
    ["plot_public_data.save_invalid_inputs", "reproduced_figures/invalid_input_frequency.png"],
    ["plot_public_data.save_invalid_input_heatmap", "reproduced_figures/invalid_input_heatmap.png"],
    ["plot_public_data.save_questionnaire_background_en", "reproduced_figures/questionnaire_background_en.png"],
    ["plot_public_data.save_questionnaire_background_it", "reproduced_figures/questionnaire_background_it.png"],
    ["plot_public_data.save_most_difficult_topic", "reproduced_figures/most_difficult_topic.png"],
    ["plot_public_data.save_reported_difficulties", "reproduced_figures/reported_difficulties.png"],
    ["plot_public_data.save_appendix_d_survey_results", "reproduced_figures/appendix_d_survey_results.png"],
], columns=["Function", "Output file"])
display(file_map)


## 1. Inspect the public aggregate inputs

These files are the inputs to the public plotting functions.


In [ ]:
public_files = sorted(PUBLIC.rglob("*.csv"))
pd.DataFrame({
    "file": [str(path.relative_to(ROOT)) for path in public_files],
    "rows": [len(pd.read_csv(path)) for path in public_files],
    "columns": [", ".join(pd.read_csv(path, nrows=0).columns) for path in public_files],
})


In [ ]:
activity = pd.read_csv(PUBLIC / "activity_summary.csv")
facility = pd.read_csv(FIGURE_DATA / "facility_index_by_question.csv")
status = pd.read_csv(FIGURE_DATA / "response_status_by_quiz_and_question_long.csv")
invalid = pd.read_csv(FIGURE_DATA / "invalid_inputs_by_quiz_and_question.csv")
reported = pd.read_csv(PUBLIC / "reported_difficulties_summary.csv")
display(activity.head())
display(facility.head())
display(reported.head())


## 2. Run the plotting functions

Each function below reads the aggregate inputs and writes one PNG. The functions are imported from `scripts/plot_public_data.py`, so the notebook executes repository code rather than copying it.


In [ ]:
from scripts.plot_public_data import (
    save_activity, save_activity_participation_performance,
    save_facility, save_status, save_response_status_by_question,
    save_invalid_inputs, save_invalid_input_heatmap,
    save_questionnaire_background_en, save_questionnaire_background_it,
    save_most_difficult_topic, save_reported_difficulties,
    save_appendix_d_survey_results,
)

for function, filename in [
    (save_activity, "activity_mean_grade.png"),
    (save_activity_participation_performance, "activity_participation_performance.png"),
    (save_facility, "question_level_facility_index.png"),
    (save_status, "response_status_summary.png"),
    (save_response_status_by_question, "response_status_by_question.png"),
    (save_invalid_inputs, "invalid_input_frequency.png"),
    (save_invalid_input_heatmap, "invalid_input_heatmap.png"),
    (save_questionnaire_background_en, "questionnaire_background_en.png"),
    (save_questionnaire_background_it, "questionnaire_background_it.png"),
    (save_most_difficult_topic, "most_difficult_topic.png"),
    (save_reported_difficulties, "reported_difficulties.png"),
    (save_appendix_d_survey_results, "appendix_d_survey_results.png"),
]:
    function(FIGURES)
    print(f"{function.__name__} -> {FIGURES / filename}")


## 3. Display the generated figures


In [ ]:
for filename in [
    "activity_mean_grade.png",
    "activity_participation_performance.png",
    "question_level_facility_index.png",
    "response_status_summary.png",
    "response_status_by_question.png",
    "invalid_input_frequency.png",
    "invalid_input_heatmap.png",
    "questionnaire_background_en.png",
    "questionnaire_background_it.png",
    "most_difficult_topic.png",
    "reported_difficulties.png",
    "appendix_d_survey_results.png",
]:
    path = FIGURES / filename
    print(filename)
    display(Image(filename=str(path)))


## Important limitation

`stack_analysis.pipeline.run_all_quizzes()` is the original full workflow. It calls functions such as `clean_quiz_responses`, `build_response_level_table`, `build_analysis_df`, and the analysis helpers, then writes student-level outputs under `outputs/`. Those inputs and outputs are intentionally excluded from this public notebook and repository release.

Figures 1–3 in the paper are source artwork (a STACK screenshot, a conceptual framework, and a workflow diagram), not numerical-analysis outputs. They should be preserved as original image assets separately if the journal requests them.

The paper's Figure 8 labels the two responses recorded as `Equazioni` as `Expressions`; the data-faithful reproduction labels them `Equations`. Confirm the preferred wording before submission.

The Appendix D screenshot uses a different set of coded frequencies from the current public aggregate file. The repository plot is generated from the current public aggregate counts; matching the screenshot exactly requires the corresponding final coding version.

For the public release, the reproducible path is the aggregate-only workflow above: `public_data/` supplies the shareable inputs and `plot_public_data.save_*` supplies the data-driven figures.
